# U2OS density-gradient fractionation (LoRNA): fracSanity tested against spike-ins

**Data.** LoRNA of Villanueva et al. (Nat Methods 2024; ENA PRJEB49479): U2OS lysate loaded at the density of free RNA into an iodixanol gradient; membranes (ER, mitochondria) float up, RNA–protein complexes sediment. 8 fractions and the whole-cell lysate (`dTotal`), 3 replicates each, control (DMSO) cells only. CORALL total RNA-seq (rRNA depleted), paired-end; Lexogen SIRV spike-ins in every library. Processed with the bulk RNA-seq workflow (run `U2OS_subcellular_fractions`, configuration in `NFYA_project.ipynb`: STAR, featureCounts on GENCODE v42 basic, all genes) and, for the spike-ins, STAR on the SIRV (and ERCC) sequences (`WF/scripts/count_spike_ins.sbatch`). Fraction identities as in `NFYA_project.ipynb` (protein markers of the paper): 1–2 ER + mitochondria, 4 nucleolus, 6–8 membrane-less cytosol, 3 and 5 unclear there; the spike-in $\alpha$ of marker RNAs below places 3 with 1–2 (ER mRNAs and mitochondrial transcripts peak in 3) and 5 with 4 (MALAT1, NEAT1, 7SK, snoRNAs: nuclear RNPs).

**Why this data set.** fracSanity models the fraction $\alpha_g$ of the molecules of gene $g$ that are in a sub-fraction, with $D_g = \log_2\rho_g^{\text{fraction}} - \log_2\rho_g^{\text{Total}} = \log_2\alpha_g - \log_2 F$, where $F$ is the share of all RNA molecules in the sub-fraction. From sequencing alone, $F$ is not identified (FACS-sorted organelles, P-bodies). Here, the authors used the SIRV spike-ins to measure the RNA content of every fraction (Methods, "Estimation of localization proportions": *"RNA content per fraction was estimated using the relative abundance of all SIRV features and human RNA and the proportion of the fraction used for RNA-seq library preparation"*). With it, $\alpha_g$ is known up to the measurement noise of $D_g$:
$$\alpha^{T}_{g,k} = F^{\text{spike}}_k\, 2^{D_{g,k}} \quad\text{(relative to the Total library, the quantity fracSanity estimates)}, \qquad \alpha^{\Sigma}_{g,k} = \frac{F^{\text{spike}}_k\,\rho_{g,k}}{\sum_j F^{\text{spike}}_j\,\rho_{g,j}} \quad\text{(relative to the sum of the fractions; no Total library needed)}.$$
The 8 fractions are 8 independent tests of fracSanity, from a few % to ~30% of the RNA.

**How the spike-ins give the RNA content.** Methods, "RNA-seq": *"libraries for the density fractionation were generated using 1 μg of RNA as starting material and depleting ribosomal RNA [...]. Post ribosomal RNA depletion, RNA content was measured with a Bioanalyzer pico kit (Agilent). Then 1% RNA spike-in RNA variants (SIRVs) were spiked-in"*. So the SIRVs were **not** added in the same amount to every fraction: each library got SIRVs in proportion to its rRNA-depleted RNA (the authors' table: 0.02–0.12 ng per library, about 0.5% of the measured depleted RNA), and every library started from 1 µg of the fraction's RNA, whatever the RNA yield of the fraction. The authors' code (GitHub `CambridgeCentreForProteomics/LoRNA_UPR`, `1_estimate_spike_in.Rmd`) combines three numbers per library: the SIRV share of the reads $s_k$, the ng SIRVs added $m_k$ and the total RNA of the fraction $R_k$ (1 µg / proportion of the fraction used):
$$\text{non-rRNA}_k = \frac{m_k}{s_k}\cdot\frac{R_k}{1\,\mu\text{g}}, \qquad F^{\text{spike}}_k = \frac{\text{non-rRNA}_k}{\sum_{j=1}^{8}\text{non-rRNA}_j}.$$
$m_k$ and $R_k$ are not in the reads; the authors' processed table (`sirv_correction.rds`, copied to `metadata/LoRNA_UPR_Villanueva2024/` with its provenance) has them. From the reads alone one gets only $K_k = \sum_g (c_{g,k}/L_g)/S_k$ (human molecules per SIRV read; $c$ read pairs, $L$ gene length, $S$ SIRV read pairs) and $F^{K}_k = K_k/\sum_j K_j$, which equals the RNA content only if every library got the same SIRV amount, which is not the case. Both are shown; $F^{\text{spike}}$ (the authors') is the reference.

**Independent check: the mass balance.** If the fractions together contain the lysate, then $\rho_g^{\text{Total}} = \sum_k F_k\rho_{g,k}$ for every gene, which determines $F_k$ from the expression data alone (non-negative least squares over the well-expressed genes), best on the 3 pooled compartments (membranes 1–3, nuclear 4–5, cytosol 6–8), whose profiles are nearly independent.

**What is tested.**
1. $F$: fracSanity posterior (uniform prior) vs $F^{\text{spike}}$ and $F^{\text{mb}}$; profile likelihoods; per-library fits against the Total of the same replicate.
2. $\alpha_g$: posterior median, credible intervals and the F-free ranking $r_g$ against $\alpha^{T}$ and $\alpha^{\Sigma}$; with a uniform prior on $F$, with a prior from the mass balance (a way to fix $F$ without spike-ins), with a tight prior around $F^{\text{spike}}$ (the gene-level part of the model when $F$ is known) and, for comparison, around the read-only $F^{K}$ (first version of this analysis).
3. The model assumptions: Beta distribution of $\alpha_g$ (against the spike-in $\alpha^T$), outlier component, consistency over the 8 fractions ($\sum_k\alpha_{g,k} = 1$).
4. Comparison of the fractions with the FACS-sorted ONT fractions (HCT116) and APEX-seq (HEK293T).

Pipeline: `WF/scripts/LoRNA_fracSanity_pipeline.py` (`sbatch LoRNA_fracSanity.sbatch`); Sanity (EAP) on genes with gene lengths (log2 TPM: reads come from along the transcripts), Total and fractions separately, as for the ONT and APEX-seq data.

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import re
import csv
import pickle

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr, beta as beta_dist

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"wf_runs_dir": os.getenv("WF_RUNS_DIR"), "figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR")}
ANALYSIS = "LoRNA_U2OS_fracSanity"
SU_OUT = os.path.join(subdirs["wf_runs_dir"], "Sanity_utils_LoRNA_U2OS")      # WF/scripts/LoRNA_fracSanity_pipeline.py
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)
T_ONT = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", "ONT_organelles_fracSanity")
T_APEX = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", "APEX_seq_fracSanity")

def figure_caption(path, formats=("png", "pdf")):
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, subdir, name, formats=("png", "pdf"), dpi=300):
    out = os.path.join(FIGS, subdir)
    os.makedirs(out, exist_ok=True)
    for fmt in formats:
        fig.savefig(os.path.join(out, f"{name}.{fmt}"), bbox_inches="tight", dpi=dpi)
    figure_caption(os.path.join(out, name), formats)

def save_tsv(df, subdir, name, index=False):
    out = os.path.join(TABLES, subdir)
    os.makedirs(out, exist_ok=True)
    df.to_csv(os.path.join(out, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)

def load_pickle(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)

def safe(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name)

COHORTS = [f"F{k}" for k in range(1, 9)]
# fractions 3 and 5 named from the spike-in alpha of marker RNAs (section "Spike-in alpha_g"); NFYA_project.ipynb: "unclear"
LABELS = {"F1": "ER+mito (1)", "F2": "ER+mito (2)", "F3": "ER+mito (3)", "F4": "nucleolus (4)", "F5": "nuclear RNPs (5)",
          "F6": "cytosol (6)", "F7": "cytosol (7)", "F8": "cytosol (8)"}
FRAC_COLORS = dict(zip(COHORTS, sns.color_palette("viridis", 8)))
REP_MARKERS = {1: "o", 2: "s", 3: "^"}
# "low pi": the outlier fraction of the error model bounded at 0.02 instead of 0.5 (genes in the tails of D_g are not discounted as outliers)
# "spike-in prior": tight prior around the authors' F_spike; "F^K prior": around the read-only F^K (equal SIRV amounts assumed)
FIT_SETS = {"uniform prior": "cohorts", "mass-balance prior": "cohorts_mbprior", "spike-in prior": "cohorts_authprior",
            "uniform prior / low pi": "cohorts_lowpi", "spike-in prior / low pi": "cohorts_authprior_lowpi",
            "F^K prior": "cohorts_spikeprior", "F^K prior / low pi": "cohorts_spikeprior_lowpi"}
FIT_COLORS = {"uniform prior": "tab:grey", "mass-balance prior": "tab:orange", "spike-in prior": "tab:blue",
              "uniform prior / low pi": "tab:green", "spike-in prior / low pi": "tab:purple", "F^K prior": "tab:cyan",
              "F^K prior / low pi": "tab:pink"}
MIN_TPM = 10           # genes used for the evaluation: Total average >= 10 TPM
os.makedirs(FIGS, exist_ok=True); os.makedirs(TABLES, exist_ok=True)
print(SU_OUT)

## Samples and reads

In [ ]:
meta = pd.read_csv(os.path.join(SU_OUT, "inputs", "metadata.tsv"), sep="\t")
genes_df = pd.read_csv(os.path.join(SU_OUT, "inputs", "genes.tsv"), sep="\t", index_col=0)
gene_names = genes_df["gene_name"].fillna(pd.Series(genes_df.index, index=genes_df.index))
biotype = genes_df["gene_type"].where(genes_df["chrom"] != "chrM", "mitochondrially_encoded")
sp = pd.read_csv(os.path.join(SU_OUT, "inputs", "spike_ins.tsv"), sep="\t", index_col=0)
mb = pd.read_csv(os.path.join(SU_OUT, "inputs", "mass_balance.tsv"), sep="\t")
qc = meta.set_index("sample").join(sp[["SIRV_pairs", "ERCC_pairs", "SIRV_share", "K", "F_spike"]].rename(columns={"F_spike": "F_K"}))
auth = pd.read_csv(os.path.join(SU_OUT, "inputs", "F_authors.tsv"), sep="\t").set_index("sample")   # pipeline step authors_content
qc = qc.join(auth[["ng SIRVs", "fraction_sirv", "ng_RNA", "fraction_non_rRNA", "ng_non_rRNA", "F_auth"]])
qc["assigned_share"] = qc["assigned_read_pairs"] / qc["input_reads"]
qc["no_feature_share"] = qc["no_feature_read_pairs"] / (qc["assigned_read_pairs"] + qc["no_feature_read_pairs"])
save_tsv(qc.reset_index(), "samples", "libraries.tsv")
display(qc[["library", "label", "replicate", "input_reads", "uniquely_mapped_reads", "assigned_read_pairs", "no_feature_share", "SIRV_pairs",
            "SIRV_share", "ERCC_pairs"]].style.format({"input_reads": "{:,.0f}", "uniquely_mapped_reads": "{:,.0f}", "assigned_read_pairs": "{:,.0f}",
                                                      "no_feature_share": "{:.2f}", "SIRV_pairs": "{:,.0f}", "SIRV_share": "{:.4f}", "ERCC_pairs": "{:,.0f}"}))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
order = qc.index.tolist()
cols = [FRAC_COLORS.get(c, "black") for c in qc["cohort"]]
x = np.arange(len(order))
axes[0].bar(x, qc["assigned_read_pairs"] / 1e6, color=cols, edgecolor="black", lw=0.3)
axes[0].set(ylabel="assigned read pairs (M)")
axes[1].bar(x, qc["no_feature_share"], color=cols, edgecolor="black", lw=0.3)
axes[1].set(ylabel="share of read pairs\noutside annotated exons")
axes[2].bar(x, 100 * qc["SIRV_share"], color=cols, edgecolor="black", lw=0.3)
axes[2].set(ylabel="SIRV read pairs (% of input)")
for ax in axes:
    ax.set_xticks(x, order, rotation=90, fontsize=6)
sns.despine(); fig.tight_layout()
savefig(fig, "QC", "reads_per_library")
plt.show()

## Spike-ins: RNA content of the fractions

**Panels:** (a) $K$ per library, human RNA per SIRV read ($\log_2$, relative to the median library); (b) composition of the SIRV reads over the 7 SIRV genes (the same mix in every library should give the same composition); (c) the authors' per-library numbers: ng SIRVs added against the non-rRNA RNA of the 1 µg input that they imply ($m_k/s_k$; a constant ratio means the SIRVs were dosed in proportion to the depleted RNA); (d) $F^{\text{spike}}_k$ (authors) and $F^{K}_k$ (reads only) per replicate. The Total library has no $F$ (it is not part of the gradient); its $K$ is shown for comparison. Our SIRV share (STAR read pairs on the SIRV sequences / input pairs) and the authors' (Salmon TPM share) differ by a factor of about 2 but correlate over the libraries.

In [ ]:
pg = pd.read_csv(os.path.join(SU_OUT, "inputs", "spike_ins.per_gene.tsv"), sep="\t", index_col=0)
sirv = pg.loc[pg.index.str.startswith("SIRV")]
sirv_comp = sirv / sirv.sum()
comp_cv = (sirv_comp.std(axis=1) / sirv_comp.mean(axis=1)).rename("CV over libraries")
display(sirv_comp.T.join(qc["cohort"]).groupby("cohort").mean().T.round(3).join(comp_cv.round(3)))

fr = qc.loc[qc["cohort"] != "Total"]
F_sp = fr.pivot(index="cohort", columns="replicate", values="F_auth").loc[COHORTS]      # the authors' RNA content: the reference
F_K = fr.pivot(index="cohort", columns="replicate", values="F_K").loc[COHORTS]          # reads only, equal SIRV amounts assumed
K_tab = qc.pivot(index="cohort", columns="replicate", values="K").loc[["Total"] + COHORTS]
save_tsv(F_sp.add_prefix("F_spike_R").join(F_K.add_prefix("F_K_R")).join(K_tab.add_prefix("K_R")).rename_axis("cohort").reset_index(),
         "spike_ins", "F_spike_per_fraction.tsv")
save_tsv(fr[["cohort", "replicate", "SIRV_share", "fraction_sirv", "ng SIRVs", "fraction_non_rRNA", "ng_RNA", "ng_non_rRNA", "F_auth", "K", "F_K"]]
         .reset_index(), "spike_ins", "authors_RNA_content_per_library.tsv")
display(pd.DataFrame({"F_spike (authors) mean": F_sp.mean(axis=1), "sd": F_sp.std(axis=1), "F_K (reads only) mean": F_K.mean(axis=1),
                      "sd ": F_K.std(axis=1), "total RNA (ng)": fr.groupby("cohort")["ng_RNA"].mean(),
                      "non-rRNA share of the input": fr.groupby("cohort")["fraction_non_rRNA"].mean(),
                      "ng SIRVs added": fr.groupby("cohort")["ng SIRVs"].mean()}).loc[COHORTS].round(4))
print(f"Spearman of the SIRV share (ours vs the authors') over the 24 fraction libraries: {spearmanr(fr['SIRV_share'], fr['fraction_sirv'])[0]:.2f}")

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 4, figsize=(21, 3.8), gridspec_kw={"width_ratios": [1, 1.3, 1, 1]})
ax = axes[0]
for (c, r), v in qc.set_index(["cohort", "replicate"])["K"].items():
    xi = (["Total"] + COHORTS).index(c)
    ax.scatter(xi + (r - 2) * 0.12, np.log2(v / qc["K"].median()), color=FRAC_COLORS.get(c, "black"), marker=REP_MARKERS[r], s=36,
               edgecolor="black", lw=0.4)
ax.set_xticks(range(9), ["Total"] + [f"{c}\n{LABELS[c]}" for c in COHORTS], rotation=90, fontsize=7)
ax.set(ylabel="log2 K (human RNA per SIRV,\nrelative to the median library)")
ax = axes[1]
bottom = np.zeros(sirv_comp.shape[1])
for i, g in enumerate(sirv_comp.index):
    ax.bar(np.arange(sirv_comp.shape[1]), sirv_comp.loc[g], bottom=bottom, color=sns.color_palette("Set2", 8)[i], label=g, lw=0)
    bottom += sirv_comp.loc[g].to_numpy()
ax.set_xticks(np.arange(sirv_comp.shape[1]), [qc.index[qc["library"] == l][0] if (qc["library"] == l).any() else l for l in sirv_comp.columns],
              rotation=90, fontsize=6)
ax.set(ylabel="share of SIRV read pairs", ylim=(0, 1))
ax.legend(fontsize=6, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
ax = axes[2]
for (c, r), d in fr.set_index(["cohort", "replicate"]).iterrows():
    ax.scatter(1000 * d["fraction_non_rRNA"], d["ng SIRVs"], color=FRAC_COLORS[c], marker=REP_MARKERS[r], s=36, edgecolor="black", lw=0.4)
ax.set(xscale="log", yscale="log", xlabel="non-rRNA in the 1 µg library input (ng)", ylabel="SIRVs added (ng)")
ax.legend(handles=[plt.Line2D([], [], color=FRAC_COLORS[c], marker="o", ls="", label=c) for c in COHORTS], fontsize=6, frameon=False)
ax = axes[3]
for r in F_sp.columns:
    ax.plot(range(8), F_sp[r], marker=REP_MARKERS[r], color="black", lw=0.8, label=f"$F^{{spike}}$ (authors), replicate {r}")
    ax.plot(range(8), F_K[r], marker=REP_MARKERS[r], color="grey", lw=0.6, ls="--", label=f"$F^K$ (reads only), replicate {r}")
ax.set_xticks(range(8), COHORTS)
ax.set(ylabel="share of the gradient's RNA", xlabel="fraction (top → bottom of the gradient)")
ax.legend(fontsize=6, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "spike_ins", "spike_ins.K_composition_F")
plt.show()

### How the authors obtain the RNA content, and how it compares with equal spike-in amounts added before RNA isolation

**The authors' procedure, step by step** (Methods "Density-based cell fractionation", "RNA and protein sample precipitation", "RNA-seq", "Estimation of localization proportions"; code `1_estimate_spike_in.Rmd` and `3_Normalise_by_abundance.Rmd` of `CambridgeCentreForProteomics/LoRNA_UPR`, copied to `metadata/LoRNA_UPR_Villanueva2024/`):
1. The gradient is collected as 20 fractions of 500 µl. Each is adjusted to 30% iodixanol, dried by vacuum centrifugation and dissolved in TRIzol. RNA is extracted (chloroform, isopropanol, ethanol washes), DNase-treated on RNeasy columns and quantified by UV absorbance (DS-11 spectrophotometer). The 20 fractions are pooled into the 8 sequenced ones (Supplementary Table 4 of the paper).
2. From each of the 8 fractions, **1 µg** of RNA (a share $p_k$ of the fraction's RNA) goes into the library. The total RNA of the fraction is $R_k = 1\,\mu\text{g}/p_k$ (in the authors' code `ng_RNA` $= 2000/p_k$; the 2 µg are a constant and cancel in $F$).
3. rRNA is depleted (RiboCop), the depleted RNA is measured on a Bioanalyzer (pico kit), and **SIRVs are added at about 1% of that amount**: $m_k$ = 0.02–0.12 ng per library (table above). The amount follows the depleted RNA of each library, not the fraction.
4. After sequencing, the SIRV share of the reads $s_k$ (Salmon TPM of the SIRVs / all TPM) gives the non-rRNA RNA in the library, $m_k/s_k$ (the "1%" is re-measured: errors of the Bioanalyzer and of pipetting a few pg of SIRVs are corrected, as long as $m_k$ is the amount actually added). Per µg of input: $f_k = m_k/(s_k \cdot 1\,\mu\text{g})$ (`fraction_non_rRNA`, 0.7–2.7%).
5. The non-rRNA RNA of the whole fraction is $f_k R_k$ (`ng_non_rRNA`); the authors multiply the TPM of fraction $k$ by it and normalize each gene over the 8 fractions. For fracSanity, $F^{\text{spike}}_k = f_k R_k / \sum_j f_j R_j$.

So the spike-ins measure only the **non-rRNA share of the 1 µg input**. The RNA yield of the fraction, which carries most of the differences between fractions (5 µg in fraction 5 against 96 µg in fraction 7; the non-rRNA share differs only 4-fold), comes from the UV measurement in step 1.

**Alternative: the same amount of spike-in RNA added to every sample before RNA isolation** (as in our other studies, e.g. the same volume of spike-in mix per fraction, or per number of cell equivalents, added to the lysate or to TRIzol). Then the reads give the content directly: $F_k \propto K_k = (\text{human molecules}/\text{spike molecules})_k$, the quantity $F^K$ computed above, but valid because the assumption behind it holds by design.

| | authors: spikes in proportion to the library input, plus external yields | equal spike amounts per sample before RNA isolation |
|:---|:---|:---|
| what the spike controls | rRNA depletion, library preparation, sequencing (from step 3 on) | everything from the point of addition: extraction, precipitation, DNase/column clean-up, rRNA depletion, library preparation, sequencing |
| what must be measured outside the reads | RNA yield of each fraction (UV), the share used (1 µg), the Bioanalyzer amount and the SIRV amount added | the share of each sample processed (if not the whole sample), cell equivalents per sample |
| losses during extraction and clean-up | **not corrected**: they enter $R_k$, and can differ between fractions (membrane- and lipid-rich fractions 1–3 vs protein-rich 6–8; residual iodixanol, which absorbs in the UV, could also bias the A260) | corrected, as far as the spike is extracted like the endogenous RNA |
| units | ng of non-rRNA RNA (mass, including intronic, intergenic and small RNAs); fracSanity's $F$ is a share of molecules of the analysed genes, so a conversion by mean length and genic share is needed (below) | molecules, the same units as TPM / CPM, when spike reads are length-normalized like the genes |
| spike read share | the same (~0.2–0.6% of reads) in every library: precise $s_k$ everywhere | proportional to 1/(RNA of the sample): RNA-poor samples can be dominated by spike reads and RNA-rich samples have few, which is noisy unless the amount is tuned to the expected yields |
| library input | equal (1 µg) in every library: comparable complexity | as much as the sample has; low-yield samples give low-complexity libraries (see the compression analyses of the ONT and P-body data) |
| reference to the whole cell | only relative over the 8 fractions ($\sum_k F_k = 1$); losses of the gradient as a whole are invisible | absolute against a Total sample with spikes per cell equivalent: $F$ of a single sorted compartment (ONT organelles, P-bodies) relative to the whole cell, without needing the other fractions |
| main risks | errors of UV quantification at low yields; differential recovery; pipetting of pg amounts of SIRVs (corrected only if the recorded $m_k$ is right) | the spike (free, naked RNA) may be extracted or degraded differently from RNA inside organelles or RNPs; spike must be added after RNase inactivation (e.g. into TRIzol) |

In short: the authors' design keeps libraries uniform and spike reads precise, but the step that decides most of $F$ (the RNA yield of each fraction) is an external UV measurement whose losses are not controlled. Adding the same spike amount before isolation controls those losses and gives $F$ in molecules directly, and it is the only way to get $F$ for a single sorted compartment against the whole cell; its cost is unequal spike read shares and library inputs.

**Checks in these data.** (i) $F^{\text{spike}}$ agrees with the mass balance on the pooled compartments (next sections), which needs neither spikes nor yields; (ii) the conversion from mass to molecules of the analysed genes (below) changes single fractions by up to ~25%, which bounds the uncertainty of the reference from the unit mismatch.

In [ ]:
# F^spike (non-rRNA mass) converted to a share of molecules of the analysed genes: mass ~ read pairs (CORALL reads cover the
# whole RNA), so molecules of the annotated genes ~ mass x (share of read pairs on annotated exons) / (molecule-weighted mean length)
counts_raw = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_counts.tsv.gz"), sep="\t", index_col=0)
glen = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_lengths.tsv"), sep="\t", index_col=0)["length"].reindex(counts_raw.index)
mol = counts_raw.loc[glen.notna()].div(glen.dropna(), axis=0)
fr = fr.assign(mean_length=(counts_raw.loc[glen.notna()].sum() / mol.sum()).reindex(fr.index),
               genic_share=fr["assigned_read_pairs"] / (fr["assigned_read_pairs"] + fr["no_feature_read_pairs"]))
w_mol = fr["F_auth"] * fr["genic_share"] / fr["mean_length"]
fr["F_auth_molecules"] = w_mol / w_mol.groupby(fr["replicate"]).transform("sum")
F_sp_mol = fr.pivot(index="cohort", columns="replicate", values="F_auth_molecules").loc[COHORTS]
conv = pd.DataFrame({"F_spike (mass)": F_sp.mean(axis=1), "F_spike (molecules of the analysed genes)": F_sp_mol.mean(axis=1),
                     "mean length (nt)": fr.groupby("cohort")["mean_length"].mean(), "genic share of read pairs": fr.groupby("cohort")["genic_share"].mean()})
conv["log2 molecules / mass"] = np.log2(conv.iloc[:, 1] / conv.iloc[:, 0])
save_tsv(conv.rename_axis("cohort").reset_index(), "spike_ins", "F_spike_mass_vs_molecules.tsv")
display(conv.round(3))

## Sanity normalization and the mass balance

Sanity (EAP, log2 TPM) on the 3 Total and the 24 fraction libraries separately. Mass balance: $1 = \sum_k F_k\,\rho_{g,k}/\rho_{g}^{\text{Total}}$ for every gene with Total ≥ 10 TPM, solved for $F_k \ge 0$ by non-negative least squares, on the cohort averages (95% bootstrap interval over genes) and per replicate (fraction libraries against the Total library of the same replicate). $\sum_k F^{\text{mb}}_k$ close to 1 means that the fractions together recover the composition of the lysate.

**Panels:** (a) $F^{\text{mb}}$ vs $F^{\text{spike}}$ per fraction and replicate (log scale); (b) the profile of both over the gradient (averages); (c) per gene: $\log_2\sum_k F^{\text{spike}}_k\,\rho_{g,k}/\rho_g^{\text{Total}}$, the recovery of gene $g$ in the gradient relative to the Total library, per biotype (0 = recovered as the average RNA).

In [ ]:
tot = load_pickle(os.path.join(SU_OUT, "sanity", "genes", "Total", "sanity.pkl"))
frac = load_pickle(os.path.join(SU_OUT, "sanity", "genes", "fractions", "sanity.pkl"))
tot_av, frac_av = tot["averages"], frac["averages"]
for part, d in (("Total", tot), ("fractions", frac)):
    r = d["result"]
    print(f"{part:9s}: {r.log2_expression.shape[0]:,} genes x {r.log2_expression.shape[1]} libraries | {r.params.get('units')} | v_method {r.params.get('v_method')}")
G = tot_av.means.index.intersection(frac_av.means.index)
mT, eT = tot_av.means.loc[G, "Total"], tot_av.errors.loc[G, "Total"]
M, E = frac_av.means.loc[G, COHORTS], frac_av.errors.loc[G, COHORTS]
total_tpm = 2 ** mT
EVAL = G[(total_tpm >= MIN_TPM).to_numpy()]
print(f"{len(G):,} genes in Total and fractions; {len(EVAL):,} with Total >= {MIN_TPM} TPM (evaluation set)")

F_spike = F_sp.mean(axis=1)
mb_av = mb.loc[mb["replicate"] == "average"].set_index("cohort").loc[COHORTS]
mb_rep = mb.loc[mb["replicate"] != "average"].assign(replicate=lambda d: d["replicate"].astype(int)).pivot(index="cohort", columns="replicate",
                                                                                                         values="F_mb").loc[COHORTS]
F_tab = pd.DataFrame({"label": pd.Series(LABELS), "F_spike": F_spike, "F_spike_sd": F_sp.std(axis=1), "F_mb": mb_av["F_mb"],
                      "F_mb_CI_lower": mb_av["F_mb_CI_lower"], "F_mb_CI_upper": mb_av["F_mb_CI_upper"]}).loc[COHORTS]
print(f"sum of F_mb over the 8 fractions: {F_tab['F_mb'].sum():.3f} (average); per replicate "
      + ", ".join(f"R{r} {mb_rep[r].sum():.3f}" for r in mb_rep.columns))
display(F_tab.round(4))

# recovery of each gene in the gradient relative to Total, with the spike-in F
recovery = np.log2((2 ** M).mul(F_spike, axis=1).sum(axis=1) / 2 ** mT)
rec = pd.DataFrame({"gene_name": gene_names.reindex(G), "biotype": biotype.reindex(G), "Total_TPM": total_tpm, "log2_recovery": recovery})
save_tsv(rec.rename_axis("gene_id").reset_index(), "mass_balance", "gene_recovery_in_gradient.tsv.gz")

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1, 1, 1.4]})
ax = axes[0]
for c in COHORTS:
    for r in F_sp.columns:
        ax.scatter(F_sp.loc[c, r], mb_rep.loc[c, r], color=FRAC_COLORS[c], marker=REP_MARKERS[r], s=36, edgecolor="black", lw=0.4)
lim = [min(F_sp.min().min(), mb_rep.replace(0, np.nan).min().min()) / 1.5, max(F_sp.max().max(), mb_rep.max().max()) * 1.5]
ax.plot(lim, lim, color="grey", lw=0.8, ls="--")
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="$F^{spike}$", ylabel="$F^{mb}$ (mass balance)")
ax.legend(handles=[plt.Line2D([], [], color=FRAC_COLORS[c], marker="o", ls="", label=f"{c} {LABELS[c]}") for c in COHORTS],
          fontsize=6, frameon=False, loc="upper left")
ax = axes[1]
ax.errorbar(range(8), F_tab["F_spike"], yerr=F_tab["F_spike_sd"], marker="o", color="tab:blue", label="$F^{spike}$ (mean ± sd)", capsize=2)
ax.errorbar(np.arange(8) + 0.1, F_tab["F_mb"], yerr=[F_tab["F_mb"] - F_tab["F_mb_CI_lower"], F_tab["F_mb_CI_upper"] - F_tab["F_mb"]],
            marker="s", color="tab:orange", label="$F^{mb}$ (95% bootstrap)", capsize=2)
ax.set_xticks(range(8), COHORTS)
ax.set(ylabel="share of the RNA", xlabel="fraction")
ax.legend(fontsize=7, frameon=False)
ax = axes[2]
ev = rec.loc[EVAL]
top_bt = ev["biotype"].value_counts().index[:9]
sns.boxplot(data=ev.loc[ev["biotype"].isin(top_bt)], y="biotype", x="log2_recovery", order=top_bt, showfliers=False, ax=ax, color="lightgrey")
ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel="log2 recovery in the gradient vs Total\n(with $F^{spike}$)", ylabel="")
ax.set_yticks(range(len(top_bt)), [f"{b} ({(ev['biotype'] == b).sum():,})" for b in top_bt], fontsize=7)
sns.despine(); fig.tight_layout()
savefig(fig, "mass_balance", "F_mass_balance_vs_spike")
plt.show()
display(rec.loc[EVAL].groupby("biotype")["log2_recovery"].agg(["size", "median"]).sort_values("size", ascending=False).head(10).round(3))

### Does $F^{\text{spike}}$ describe the RNA content of the fractions?

Two checks:
1. Does the read-only $K$ differ between fractions more than between replicates (two-way ANOVA of $\log_2 K$ on fraction and replicate)? With SIRVs dosed in proportion to the depleted RNA of each library, $K$ should be nearly constant, and the RNA content must come from the amounts the authors recorded.
2. Does $F$ explain the composition of the lysate? Gene-wise mass balance per replicate, $\log_2\sum_k F_k\rho_{g,k}/\rho_g^{\text{Total}}$ (genes ≥ 10 TPM in that Total library), for: the authors' $F^{\text{spike}}$ of the replicate, of another replicate and permuted over the fractions; the read-only $F^K$; $F = 1/8$ for every fraction; and $F^{\text{mb}}$ from NNLS on a random half of the genes (evaluated on the other half; fitted to these data, so the best attainable). A correct $F$ gives a small spread across genes (median absolute deviation, MAD; share within 1.5× of the median); the median is below 0 when RNA is lost in the gradient.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
kd = qc.loc[qc["cohort"] != "Total", ["cohort", "replicate", "K"]].assign(log2K=lambda d: np.log2(d["K"]))
anova = sm.stats.anova_lm(smf.ols("log2K ~ C(cohort) + C(replicate)", data=kd).fit(), typ=2)
display(anova.round(4))
print(f"K: CV over the 24 fraction libraries {kd['K'].std() / kd['K'].mean():.2f}, mean CV within a fraction "
      f"{kd.groupby('cohort')['K'].agg(lambda x: x.std() / x.mean()).mean():.2f}; Total libraries CV "
      f"{qc.loc[qc['cohort'] == 'Total', 'K'].std() / qc.loc[qc['cohort'] == 'Total', 'K'].mean():.2f}")

from scipy.optimize import nnls
lt, lf = tot["result"].log2_expression, frac["result"].log2_expression
fm = frac["metadata"]
rng = np.random.default_rng(0)
mbt_rows = []
for _, t in tot["metadata"].iterrows():
    rep = int(t["replicate"])
    libs = [fm.loc[(fm["cohort"] == c) & (fm["replicate"] == rep), "sample"].iloc[0] for c in COHORTS]
    g = lt.index.intersection(lf.index)
    g = g[(lt.loc[g, t["sample"]] >= np.log2(MIN_TPM)).to_numpy()]
    R = 2.0 ** (lf.loc[g, libs].to_numpy() - lt.loc[g, t["sample"]].to_numpy()[:, None])
    half = rng.random(len(g)) < 0.5
    vecs = {"F_spike, authors (this replicate)": F_sp[rep].to_numpy(), "F_spike, authors (another replicate)": F_sp[rep % 3 + 1].to_numpy(),
            "F_spike, authors (permuted over fractions)": rng.permutation(F_sp[rep].to_numpy()),
            "F_spike, authors, as molecules (this replicate)": F_sp_mol[rep].to_numpy(),
            "F_K, reads only (this replicate)": F_K[rep].to_numpy(), "F = 1/8": np.full(8, 1 / 8),
            "F_mb (NNLS on the other half of the genes)": nnls(R[half], np.ones(half.sum()))[0]}
    for k, F in vecs.items():
        s = np.log2(R[~half] @ F)
        mbt_rows.append({"replicate": rep, "F": k, "median_log2_sum": np.median(s), "MAD_log2_sum": np.median(np.abs(s - np.median(s))),
                         "share_within_1.5x_of_median": np.mean(np.abs(s - np.median(s)) < np.log2(1.5))})
mbt = pd.DataFrame(mbt_rows)
save_tsv(mbt, "mass_balance", "mass_balance_test_of_F_spike.tsv")
display(mbt.groupby("F", sort=False)[["median_log2_sum", "MAD_log2_sum", "share_within_1.5x_of_median"]].agg(["mean", "min", "max"]).round(3))

## Spike-in $\alpha_g$: where the RNAs are in the gradient

$\alpha^{\Sigma}_{g,k}$ (cohort averages, $F^{\text{spike}}$) of marker RNAs and the median per biotype: the fraction identities of the paper should be visible (mitochondrial transcripts in the membrane fractions 1–2, nucleolar RNAs in 4, mRNAs of cytosolic proteins in 6–8).

In [ ]:
w = (2 ** M).mul(F_spike, axis=1)
alpha_S = w.div(w.sum(axis=1), axis=0)                         # relative to the sum of the fractions
alpha_T = (2 ** M.sub(mT, axis=0)).mul(F_spike, axis=1)       # relative to the Total library (fracSanity's alpha)
sd_log2_T = np.sqrt(E ** 2 + (eT ** 2).to_numpy()[:, None])
truth = pd.concat({"alpha_T": alpha_T, "alpha_S": alpha_S, "sd_log2_alpha_T": sd_log2_T}, axis=1)
truth.columns = [f"{a};{c}" for a, c in truth.columns]
save_tsv(pd.DataFrame({"gene_name": gene_names.reindex(G), "biotype": biotype.reindex(G), "Total_TPM": total_tpm}).join(truth)
         .rename_axis("gene_id").reset_index(), "truth", "alpha_spike_in.per_gene.tsv.gz")

MARKERS = ["MT-CO1", "MT-ND1", "MT-RNR2", "MALAT1", "NEAT1", "XIST", "RN7SK", "RN7SL1", "RMRP", "RPPH1", "SNORD3A", "SNHG1", "CALR", "HSPA5",
           "P4HB", "SEC61B", "CANX", "SSR4", "ACTB", "GAPDH", "RPLP0", "RPS2", "EEF1A1", "FTL", "H2BC12", "H4C3"]
mk = [g for g in MARKERS if (gene_names.reindex(G) == g).any()]
idx = [G[(gene_names.reindex(G) == g).to_numpy()][0] for g in mk]
bt_groups = biotype.reindex(EVAL)
bt_top = [b for b in bt_groups.value_counts().index if (bt_groups == b).sum() >= 15][:10]
bt_med = alpha_S.loc[EVAL].groupby(bt_groups).median().loc[bt_top]

sns.set(font_scale=0.8, style="white")
fig, axes = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={"width_ratios": [1, 1]})
sns.heatmap(alpha_S.loc[idx].set_axis(mk), cmap="mako_r", vmin=0, vmax=1, annot=True, fmt=".2f", annot_kws={"fontsize": 6}, ax=axes[0],
            cbar_kws={"label": r"$\alpha^{\Sigma}_{g,k}$"})
axes[0].set_xticklabels([f"{c}\n{LABELS[c]}" for c in COHORTS], rotation=90, fontsize=7)
axes[0].set(title="marker RNAs", ylabel="")
sns.heatmap(bt_med.set_axis([f"{b} ({(bt_groups == b).sum():,})" for b in bt_top]), cmap="mako_r", vmin=0, vmax=bt_med.values.max(), annot=True,
            fmt=".2f", annot_kws={"fontsize": 6}, ax=axes[1], cbar_kws={"label": r"median $\alpha^{\Sigma}_{g,k}$"})
axes[1].set_xticklabels([f"{c}\n{LABELS[c]}" for c in COHORTS], rotation=90, fontsize=7)
axes[1].set(title=f"median per biotype (genes >= {MIN_TPM} TPM in Total)", ylabel="")
fig.tight_layout()
savefig(fig, "truth", "alpha_spike_in.markers_and_biotypes")
plt.show()

## fracSanity: $F$ against the spike-ins

fracSanity per fraction (average of the 3 replicates against the average of the 3 Total libraries), three priors on $F$: uniform (the default, as for the ONT and APEX-seq data), from the mass balance, and tight around $F^{\text{spike}}$ (95% within a factor 1.25; also around the read-only $F^K$, "F^K prior"); all with the outlier fraction of the error model bounded at $\pi \le 0.5$ (as for ONT and APEX-seq). Two more fits bound it at $\pi \le 0.02$ (**low pi**; uniform and spike-in prior): in the APEX-seq fits, $\pi$ reached its bound 0.5 and the genes flagged as outliers (error variance $V_g + \tau^2$, $\tau$ = 2–4) lost nearly all information on $\alpha_g$ (e.g. snoRNAs with $D_g \approx 5$ in the nuclear fraction got $r_g \approx 0$); the low-pi fits test whether this discounts the most strongly localised genes. **Panels:** (a) posterior median and 95% interval of $F$ per fit and the two external estimates; (b) profile likelihood of $\log_2 F$ (uniform-prior fits; NLL minus its minimum; dashed: $F^{\text{spike}}$, dotted: $F^{\text{mb}}$; the horizontal line marks 1.92, the 95% threshold).

In [ ]:
fits = {}
for k, sub in FIT_SETS.items():
    d = os.path.join(SU_OUT, "fracSanity", "genes", sub)
    if os.path.isdir(d):
        fits[k] = {c: load_pickle(os.path.join(d, f"{safe(c)}.pkl")) for c in COHORTS if os.path.exists(os.path.join(d, f"{safe(c)}.pkl"))}
print({k: len(v) for k, v in fits.items()})

F_rows = []
for k, fs in fits.items():
    for c, f in fs.items():
        prof = f["profile"]
        nll_at = lambda F: float(np.interp(np.log2(F), prof["log2F"], prof["nll"]) - prof["nll"].min())
        F_rows.append({"fit": k, "cohort": c, "F": f["F"], "F_CI_lower": f["F_CI_lower"], "F_CI_upper": f["F_CI_upper"], "F_MLE": float(f["F_MLE"]),
                       "F_identifiable": f["F_identifiable"], "a": f["a"], "b": f["b"], "pi_outlier": f["pi_outlier"], "tau": f.get("tau"),
                       "ks_stat": f["ks_stat"], "F_spike": F_spike[c], "F_mb": F_tab.loc[c, "F_mb"],
                       "log2_F_over_F_spike": np.log2(f["F"] / F_spike[c]), "spike_in_CI": f["F_CI_lower"] <= F_spike[c] <= f["F_CI_upper"],
                       "dNLL_at_F_spike": nll_at(F_spike[c]), "dNLL_at_F_mb": nll_at(max(F_tab.loc[c, "F_mb"], 1e-6)), "n_genes": len(f["gene_ids"])})
F_res = pd.DataFrame(F_rows)
save_tsv(F_res, "F", "F_fracSanity_vs_spike_in.tsv")
display(F_res.drop(columns=["tau", "ks_stat"]).round(4))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
for i, (k, fs) in enumerate(fits.items()):
    d = F_res.loc[F_res["fit"] == k].set_index("cohort").reindex(COHORTS)
    xs = np.arange(8) + (i - (len(fits) - 1) / 2) * 0.13
    ax.errorbar(xs, d["F"], yerr=[d["F"] - d["F_CI_lower"], d["F_CI_upper"] - d["F"]], fmt="o", color=FIT_COLORS[k], capsize=2, ms=4, label=f"fracSanity, {k}")
ax.scatter(np.arange(8), F_tab["F_spike"], marker="_", s=400, color="tab:red", zorder=4, label="$F^{spike}$")
ax.scatter(np.arange(8), F_tab["F_mb"], marker="_", s=400, color="black", zorder=4, label="$F^{mb}$")
ax.set_yscale("log")
ax.set_xticks(range(8), [f"{c}\n{LABELS[c]}" for c in COHORTS], fontsize=7)
ax.set(ylabel="$F$")
ax.legend(fontsize=7, frameon=False, loc="lower left")
ax = axes[1]
if "uniform prior" in fits:
    for c, f in fits["uniform prior"].items():
        prof = f["profile"]
        ax.plot(prof["log2F"], prof["nll"] - prof["nll"].min(), color=FRAC_COLORS[c], lw=1, label=c)
        ax.axvline(np.log2(F_spike[c]), color=FRAC_COLORS[c], ls="--", lw=0.8)
        ax.axvline(np.log2(max(F_tab.loc[c, "F_mb"], 1e-6)), color=FRAC_COLORS[c], ls=":", lw=0.8)
ax.axhline(1.92, color="grey", lw=0.6)
ax.set(xlabel="log2 F", ylabel="profile NLL - min", xlim=(-14, 0.2), ylim=(-0.5, None))
ax.set_yscale("symlog", linthresh=2)
ax.legend(fontsize=6, frameon=False, ncol=2)
sns.despine(); fig.tight_layout()
savefig(fig, "F", "F_fracSanity_vs_spike_in")
plt.show()

## $\alpha_g$ against the spike-in $\alpha_g$

Genes with ≥ 10 TPM in Total. Per fraction and fit: Spearman correlation of the posterior medians with $\alpha^{T}$, median of $\log_2\alpha_g - \log_2\alpha^{T}_g$ (bias; it is the error of $\log_2 F$ when the posterior follows $D_g$), median absolute error, and the share of genes whose $\alpha^{T}$ lies in the 95% credible interval (coverage; the noise of $\alpha^T$ itself, sd of $D_g$, is small for these genes but not zero). Also the F-free ranking: Spearman of $r_g = \log_2\alpha_g - \text{median}$ with the same quantity of $\alpha^{T}$, and agreement with $\alpha^{\Sigma}$, which does not use the Total library. Scatter: uniform-prior and spike-in-prior posteriors against $\alpha^T$ for each fraction.

In [ ]:
alpha_fit = {}
met_rows = []
for k, fs in fits.items():
    for c, f in fs.items():
        a = pd.DataFrame({"alpha": f["alpha_g"], "lo": f["alpha_g_CI_lower"], "hi": f["alpha_g_CI_upper"], "log2_alpha_mean": f["log2_alpha_mean"],
                          "outlier_prob": f.get("outlier_prob", np.full(len(f["gene_ids"]), np.nan))}, index=f["gene_ids"])
        alpha_fit[(k, c)] = a
        g = EVAL.intersection(a.index)
        x, t, s = np.log2(a.loc[g, "alpha"]), np.log2(alpha_T.loc[g, c]), np.log2(alpha_S.loc[g, c])
        r_fit, r_true = a.loc[g, "log2_alpha_mean"] - a.loc[g, "log2_alpha_mean"].median(), t - t.median()
        met_rows.append({"fit": k, "cohort": c, "n_genes": len(g), "spearman_vs_alpha_T": spearmanr(x, t)[0],
                         "bias_log2": float(np.median(x - t)), "MAE_log2": float(np.median(np.abs(x - t))),
                         "coverage_95": float(((alpha_T.loc[g, c] >= a.loc[g, "lo"]) & (alpha_T.loc[g, c] <= a.loc[g, "hi"])).mean()),
                         "spearman_r_vs_r_T": spearmanr(r_fit, r_true)[0], "spearman_vs_alpha_S": spearmanr(x, s)[0],
                         "bias_log2_vs_alpha_S": float(np.median(x - s)), "share_alpha_T_above_1": float((alpha_T.loc[g, c] > 1).mean()),
                         # shrinkage: OLS slope of r_g on r_g^T (1 = no shrinkage) and the genes the error model calls outliers
                         "slope_r_on_r_T": float(np.polyfit(r_true, r_fit, 1)[0]),
                         "share_outlier_prob_above_0.5": float((a.loc[g, "outlier_prob"] > 0.5).mean())})
metrics = pd.DataFrame(met_rows)
save_tsv(metrics, "alpha", "alpha_fracSanity_vs_spike_in.metrics.tsv")
display(metrics.round(3))

long = []
for (k, c), a in alpha_fit.items():
    long.append(a.rename(columns={"lo": "alpha_CI_lower", "hi": "alpha_CI_upper"}).assign(fit=k, cohort=c).rename_axis("gene_id").reset_index())
long = pd.concat(long, ignore_index=True)
long["alpha_T"] = [alpha_T.at[g, c] if g in alpha_T.index else np.nan for g, c in zip(long["gene_id"], long["cohort"])]
long["alpha_S"] = [alpha_S.at[g, c] if g in alpha_S.index else np.nan for g, c in zip(long["gene_id"], long["cohort"])]
long.insert(1, "gene_name", gene_names.reindex(long["gene_id"]).to_numpy())
save_tsv(long, "alpha", "alpha_fracSanity_and_spike_in.per_gene.tsv.gz")

show = [k for k in ("uniform prior", "spike-in prior", "spike-in prior / low pi") if k in fits]   # spike-in prior: the authors' F
sns.set(font_scale=0.8, style="ticks")
fig, axes = plt.subplots(len(show), 8, figsize=(22, 2.9 * len(show)), squeeze=False)
for i, k in enumerate(show):
    for j, c in enumerate(COHORTS):
        ax = axes[i, j]
        if (k, c) not in alpha_fit:
            ax.axis("off"); continue
        a = alpha_fit[(k, c)]
        g = EVAL.intersection(a.index)
        t, x = np.log2(alpha_T.loc[g, c]), np.log2(a.loc[g, "alpha"])
        ax.hexbin(t, x, gridsize=40, mincnt=1, bins="log", cmap="viridis", extent=(-14, 2, -14, 2))
        ax.plot([-14, 2], [-14, 2], color="red", lw=0.6)
        ax.axvline(0, color="grey", lw=0.5); ax.axhline(0, color="grey", lw=0.5)
        m = metrics.set_index(["fit", "cohort"]).loc[(k, c)]
        ax.set_title(f"{c} {LABELS[c]}\nrho {m['spearman_vs_alpha_T']:.2f}, bias {m['bias_log2']:+.2f}, cov {m['coverage_95']:.2f}", fontsize=7)
        ax.set(xlabel=r"log2 $\alpha^T$ (spike-in)" if i == len(show) - 1 else "", ylabel=f"log2 $\\alpha_g$, {k}" if j == 0 else "")
sns.despine(); fig.tight_layout()
savefig(fig, "alpha", "alpha_fracSanity_vs_spike_in.scatter")
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
for ax, col, lab in zip(axes, ["bias_log2", "coverage_95", "spearman_r_vs_r_T", "slope_r_on_r_T"],
                        ["median log2 alpha - log2 alpha^T", "coverage of alpha^T by the 95% interval", "Spearman of r_g vs r_g^T (F-free)",
                         "slope of r_g on r_g^T (1: no shrinkage)"]):
    for k in fits:
        d = metrics.loc[metrics["fit"] == k].set_index("cohort").reindex(COHORTS)
        ax.plot(range(8), d[col], marker="o", color=FIT_COLORS[k], label=k)
    ax.set_xticks(range(8), COHORTS)
    ax.set(ylabel=lab)
    if col == "bias_log2":
        ax.axhline(0, color="black", lw=0.6)
    if col == "coverage_95":
        ax.axhline(0.95, color="black", lw=0.6, ls="--")
    if col == "slope_r_on_r_T":
        ax.axhline(1, color="black", lw=0.6, ls="--")
axes[0].legend(fontsize=7, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "alpha", "alpha_metrics_per_fraction")
plt.show()

## Model assumptions

**Beta distribution of $\alpha_g$.** Density of $\log_2\alpha^{T}_g$ (genes ≥ 10 TPM in Total; spike-in $F$) and the fitted $\text{Beta}(a, b)$ of each fit, transformed to $\log_2\alpha$. The fitted Beta is the prior of the gene-level posteriors; if it is too narrow or misplaced, the posteriors are pulled towards it.

**Consistency over the 8 fractions.** fracSanity fits each fraction on its own; the $\alpha_{g,k}$ of a gene over the 8 fractions should sum to ≤ 1 (= 1 if the fractions contain the whole lysate). Distribution of $\log_2\sum_k\alpha_{g,k}$ per fit and for the spike-in $\alpha^T$.

In [ ]:
sns.set(font_scale=0.8, style="ticks")
fig, axes = plt.subplots(2, 4, figsize=(18, 6.5))
xs = np.linspace(-14, 0, 400)
for ax, c in zip(axes.flat, COHORTS):
    t = np.log2(alpha_T.loc[EVAL, c])
    ax.hist(t.clip(-14, 2), bins=np.linspace(-14, 2, 81), density=True, color="lightgrey", label=r"spike-in $\alpha^T$")
    for k, fs in fits.items():
        if c in fs:
            f = fs[c]
            dens = beta_dist.pdf(2.0 ** xs, f["a"], f["b"]) * 2.0 ** xs * np.log(2)
            ax.plot(xs, dens, color=FIT_COLORS[k], lw=1.2, label=f"Beta, {k}")
    ax.axvline(np.log2(F_spike[c]), color="tab:red", lw=0.8, ls="--")
    ax.set(title=f"{c} {LABELS[c]}", xlabel=r"log2 $\alpha$", xlim=(-14, 2))
axes.flat[0].legend(fontsize=6, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "model", "beta_prior_vs_spike_in_alpha")
plt.show()

sums = {"spike-in alpha^T": np.log2(alpha_T.loc[EVAL].sum(axis=1))}
for k, fs in fits.items():
    if len(fs) == 8:
        sums[f"fracSanity, {k}"] = np.log2(pd.DataFrame({c: alpha_fit[(k, c)]["alpha"] for c in COHORTS}).reindex(EVAL).sum(axis=1))
sum_tab = pd.DataFrame(sums)
display(sum_tab.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.round(3))
save_tsv(sum_tab.rename_axis("gene_id").reset_index(), "model", "log2_sum_alpha_over_fractions.tsv.gz")
fig, ax = plt.subplots(figsize=(6, 3.6))
for col in sum_tab.columns:
    v = sum_tab[col].dropna().clip(-6, 4)
    ax.hist(v, bins=np.linspace(-6, 4, 101), histtype="step", lw=1.2, density=True, label=col,
            color="tab:red" if col.startswith("spike") else FIT_COLORS[col.split(", ", 1)[1]])
ax.axvline(0, color="black", lw=0.6)
ax.set(xlabel=r"log2 $\sum_k \alpha_{g,k}$", ylabel="density")
ax.legend(fontsize=7, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "model", "sum_alpha_over_fractions")
plt.show()

## Why $F$ is not recovered: the shape of the $\alpha_g$ distribution

With $\alpha_g \sim \text{Beta}(a, b)$, the only information on $F$ in the $D_g$ is the **shape** of their distribution: $\log_2 F$ is a shift, and the shape of $\log\alpha$ depends on where $\alpha$ sits relative to its bound 1. The cumulants of $\log\alpha$ are $\kappa_n = \psi^{(n-1)}(a) - \psi^{(n-1)}(a+b)$ (polygamma functions), so
$$\text{skew}(\log\alpha) = \frac{\psi''(a) - \psi''(a+b)}{[\psi'(a) - \psi'(a+b)]^{3/2}} < 0 \quad\text{for all } a, b,$$
because $\psi''$ is negative and increasing. A log-Beta is **always left-skewed**. Its skewness approaches 0 only for $a \to \infty$; strong left skew with a small spread needs probability mass near $\alpha = 1$ (small $b$).

Consequences for the fit:
- $D_g$ **right-skewed** (a minority of enriched genes above the bulk): the closest log-Beta is the least skewed one, $b \to \infty$ (the Gamma limit), where the likelihood depends only on $bF$, so $F \to 0$. The enriched minority does not fit and is assigned to the outlier component.
- $D_g$ **strongly left-skewed with a small spread** (a uniform majority with a sharp upper edge, a depleted minority below): the fit places the sharp edge at $\alpha = 1$, so $F$ is overestimated.

**Panels.** (a) The attainable (standard deviation, skewness) of $\log_2\alpha$ for Beta distributions, coloured by the mean of $\alpha$, with the spike-in $\log_2\alpha^T$ of the 8 fractions and of the 3 pooled compartments; (b) skewness of $\log_2\alpha^T$ per fraction against that of the Beta fitted to $\alpha^T$ by maximum likelihood.

**Two tests** (`WF/scripts/LoRNA_fracSanity_pipeline.py`, steps `simulate` and `pooled`):
1. *Simulation with known $F$.* $D_g = \log_2\alpha_g - \log_2 F + \varepsilon_g$ with the real errors of each fraction and $F = F^{K}$ (run before the authors' $F^{\text{spike}}$ was available; any value of the right order serves, ~0.1 per fraction), with $\alpha_g$ either $F\,2^{D_g}$ from the data (the real shape of the distribution; the fit sees only the shape, so its $F$ is the same for any true $F$) or drawn from the Beta fitted to it (the model holds exactly). If $F$ is recovered under the Beta, the data are informative enough when the model holds, and the failure on real data is misspecification of the shape.
2. *Pooled compartments.* The fraction libraries of one compartment (membranes 1–3, nuclear 4–5, cytosol 6–8) merged per replicate by binomial thinning of the reads in proportion to $F^{\text{spike}}$ (step `pooled_auth`), so that each pool holds all molecules of its compartment (a gene confined to the compartment has $\alpha \approx 1$). This is the situation fracSanity was designed for (one complete compartment against the lysate); references for $F$: $\sum F^{\text{spike}}$ over the pool and the mass balance on the 3 pools (below). (A first version pooled in proportion to $F^K$, step `pooled`; its fits are listed for comparison.)

In [ ]:
from scipy.special import polygamma
from scipy.stats import skew
POOLS = {"Membranes": ["F1", "F2", "F3"], "Nuclear": ["F4", "F5"], "Cytosol": ["F6", "F7", "F8"]}
CASES = COHORTS + list(POOLS)
alpha_T_all = alpha_T.loc[EVAL].copy()
for p, mem in POOLS.items():
    alpha_T_all[p] = alpha_T_all[mem].sum(axis=1)
F_true = pd.concat([F_spike, pd.Series({p: F_spike[mem].sum() for p, mem in POOLS.items()})])

def log_beta_moments(a, b):
    # sd (log2 units) and skewness of log(alpha), alpha ~ Beta(a, b)
    k2 = polygamma(1, a) - polygamma(1, a + b)
    k3 = polygamma(2, a) - polygamma(2, a + b)
    return np.sqrt(k2) / np.log(2), k3 / k2 ** 1.5

setup_f = os.path.join(SU_OUT, "fracSanity", "genes", "simulation_setup.tsv")
setup = pd.read_csv(setup_f, sep="\t", index_col=0) if os.path.exists(setup_f) else None
shape_rows = []
for c in CASES:
    x = alpha_T_all[c].clip(1e-6, 0.999)
    a, b, _, _ = beta_dist.fit(x.to_numpy(), floc=0, fscale=1)
    sd_b, sk_b = log_beta_moments(a, b)
    shape_rows.append({"cohort": c, "F_spike": F_true[c], "share_alpha_T_above_1": float((alpha_T_all[c] > 1).mean()),
                       "median_alpha_T": x.median(), "q99_alpha_T": x.quantile(0.99),
                       "sd_log2_alpha_T": np.log2(x).std(), "skew_log2_alpha_T": skew(np.log2(x)),
                       "skew_log2_alpha_T_protein_coding": skew(np.log2(x[(biotype.reindex(x.index) == "protein_coding").to_numpy()])),
                       "beta_a": a, "beta_b": b, "sd_log2_beta": sd_b, "skew_log2_beta": sk_b})
shape = pd.DataFrame(shape_rows).set_index("cohort")
save_tsv(shape.reset_index(), "model", "shape_of_log2_alpha_T.tsv")
display(shape.round(3))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
A, B = np.meshgrid(np.logspace(-1.3, 3, 160), np.logspace(-1.3, 5, 220))
sd_g, sk_g = log_beta_moments(A, B)
sc = ax.scatter(sd_g.ravel(), sk_g.ravel(), c=np.log10(A / (A + B)).ravel(), cmap="viridis", s=2, vmin=-4, vmax=0, rasterized=True)
fig.colorbar(sc, ax=ax, label=r"log10 mean of $\alpha$ (Beta)")
for c in CASES:
    r = shape.loc[c]
    ax.scatter(r["sd_log2_alpha_T"], r["skew_log2_alpha_T"], s=60, marker="*" if c in POOLS else "o",
               color=FRAC_COLORS.get(c, "tab:red"), edgecolor="black", lw=0.6, zorder=3)
    ax.annotate(c, (r["sd_log2_alpha_T"], r["skew_log2_alpha_T"]), fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.axhline(0, color="black", lw=0.6)
ax.set(xlim=(0, 4), ylim=(-4, 2), xlabel=r"sd of log2 $\alpha$", ylabel=r"skewness of log2 $\alpha$",
       title=r"Beta distributions (dots) and spike-in $\alpha^T$ (o: fractions, *: pools)")
ax = axes[1]
x = np.arange(len(CASES))
ax.bar(x - 0.2, shape.loc[CASES, "skew_log2_alpha_T"], 0.4, color="tab:red", label=r"spike-in $\log_2\alpha^T$")
ax.bar(x + 0.2, shape.loc[CASES, "skew_log2_beta"], 0.4, color="tab:grey", label=r"Beta fitted to $\alpha^T$")
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(x, CASES, rotation=90)
ax.set(ylabel="skewness of log2 alpha")
ax.legend(fontsize=7, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "model", "skewness_of_log2_alpha")
plt.show()

### Simulation with known $F$

$F$ of fracSanity (uniform prior, default error model) on simulated $D_g$ (8,494 genes, real errors) against the true $F$ ($F^K$); $\alpha_g$ either $F^K 2^{D_g}$ (the real shape) or drawn from the Beta fitted to it. Gene level: median $\log_2\alpha_g - \log_2\alpha^{\text{sim}}_g$ (bias), coverage of $\alpha^{\text{sim}}_g$ by the 95% interval, slope of $r_g$ on $r^{\text{sim}}_g$.

In [ ]:
SIMS = {"sim_alpha_T": r"real shape ($\alpha^T$)", "sim_beta": r"Beta fitted to $\alpha^T$"}
sim_rows = []
for sub, lab in SIMS.items():
    d = os.path.join(SU_OUT, "fracSanity", "genes", sub)
    for c in CASES:
        p = os.path.join(d, safe(c) + ".pkl")
        if not os.path.exists(p):
            continue
        f = load_pickle(p)
        s = pd.read_csv(os.path.join(d, safe(c) + ".alpha_sim.tsv.gz"), sep="\t", index_col=0)["alpha_sim"]
        a = pd.Series(f["alpha_g"], index=f["gene_ids"]).reindex(s.index)
        lo = pd.Series(f["alpha_g_CI_lower"], index=f["gene_ids"]).reindex(s.index)
        hi = pd.Series(f["alpha_g_CI_upper"], index=f["gene_ids"]).reindex(s.index)
        lm = pd.Series(f["log2_alpha_mean"], index=f["gene_ids"]).reindex(s.index)
        r_fit, r_true = lm - lm.median(), np.log2(s) - np.log2(s).median()
        sim_rows.append({"simulation": sub, "cohort": c, "F_true": f["F_true"], "F": f["F"], "F_CI_lower": f["F_CI_lower"],
                         "F_CI_upper": f["F_CI_upper"], "F_identifiable": f["F_identifiable"], "pi_outlier": f["pi_outlier"],
                         "log2_F_error": np.log2(f["F"] / f["F_true"]),
                         "F_true_in_CI": bool(f["F_CI_lower"] <= f["F_true"] <= f["F_CI_upper"]),
                         "alpha_bias_log2": float(np.median(np.log2(a) - np.log2(s))),
                         "alpha_coverage_95": float(((s >= lo) & (s <= hi)).mean()),
                         "slope_r_on_r_sim": float(np.polyfit(r_true, r_fit, 1)[0])})
sim_res = pd.DataFrame(sim_rows)
if len(sim_res):
    save_tsv(sim_res, "model", "simulation_known_F.tsv")
    display(sim_res.round(3))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
    for ax, (sub, lab) in zip(axes, SIMS.items()):
        d = sim_res.loc[sim_res["simulation"] == sub]
        for _, r in d.iterrows():
            ax.errorbar(r["F_true"], r["F"], yerr=[[r["F"] - r["F_CI_lower"]], [r["F_CI_upper"] - r["F"]]],
                        fmt="*" if r["cohort"] in POOLS else "o", ms=9 if r["cohort"] in POOLS else 6,
                        color=FRAC_COLORS.get(r["cohort"], "tab:red"), mec="black", mew=0.4, capsize=2, elinewidth=0.6)
            ax.annotate(r["cohort"], (r["F_true"], r["F"]), fontsize=6, xytext=(4, 2), textcoords="offset points")
        ax.plot([1e-5, 1], [1e-5, 1], color="grey", ls="--", lw=0.8)
        ax.set(xscale="log", yscale="log", xlim=(0.03, 1), ylim=(1e-5, 1.2), title=f"simulated alpha: {lab}",
               xlabel="true F", ylabel="fracSanity F (uniform prior), 95% CI")
    sns.despine(); fig.tight_layout()
    savefig(fig, "model", "simulation_known_F")
    plt.show()
else:
    print("simulation not run yet (pipeline step simulate)")

### Pooled compartments: one complete compartment against the lysate

fracSanity (uniform prior; default and low-$\pi$ error model) on the pooled libraries against the Total. References for $F$: $F^{\text{spike}}$ summed over the fractions of the pool, and the mass balance on the 3 pooled compartments, $\rho_g^{\text{Total}} = \sum_{\text{pool}} F_{\text{pool}}\,\rho_{g,\text{pool}}$ (NNLS over genes ≥ 10 TPM; step `mass_balance_pooled`): 3 nearly independent profiles, so unlike the 8 collinear fractions the estimates are stable between replicates. Gene-level reference: $\alpha^{\text{mb}}_{g} = F^{\text{mb}}_{\text{pool}}\,2^{D_{g,\text{pool}}}$ (the $D_g$ of the pooled libraries, cohort averages). Since this reference uses the same $D_g$ as the fit, the gene-level comparison measures how the posterior distorts $D_g$ (shrinkage, outliers), and the error of $F$ adds to all genes alike.

Also: the per-fraction fits summed over the fractions of each compartment, against the same reference.

In [ ]:
POOL_FITS = {"uniform prior": "pooled_auth", "uniform prior / low pi": "pooled_auth_lowpi"}
OLD_POOL_FITS = {"uniform prior": "pooled", "uniform prior / low pi": "pooled_lowpi"}   # pools weighted by F^K (first version)
mbp_f = os.path.join(SU_OUT, "inputs", "mass_balance_pooled_auth.tsv")
pl_f = os.path.join(SU_OUT, "sanity", "genes", "pooled_auth", "sanity.pkl")
if os.path.exists(mbp_f) and os.path.exists(pl_f):
    mbp = pd.read_csv(mbp_f, sep="\t")
    display(mbp.round(4))
    F_mbp = mbp.loc[mbp["replicate"] == "average"].set_index("pool")["F_mb"]
    pl_av = load_pickle(pl_f)["averages"]
    Gp = EVAL.intersection(pl_av.means.index)
    D_pool = pl_av.means.loc[Gp, list(POOLS)].sub(tot_av.means.loc[Gp, "Total"], axis=0)
    alpha_mb = (2.0 ** D_pool).mul(F_mbp, axis=1)

    # compartment sums of the per-fraction fits against the pooled mass balance
    comp = pd.DataFrame({"F_spike (authors, sum over the fractions)": F_true[list(POOLS)], "F_mb (pooled mass balance)": F_mbp,
                         "F_mb (pooled mass balance, normalised to sum 1)": F_mbp / F_mbp.sum(),
                         "F_K (reads only, sum over the fractions)": pd.Series({p: F_K.loc[m].mean(axis=1).sum() for p, m in POOLS.items()}),
                         "F_mb (sum of the 8-fraction NNLS)": pd.Series({p: F_tab.loc[m, "F_mb"].sum() for p, m in POOLS.items()})})
    for k, sub in OLD_POOL_FITS.items():
        for p in POOLS:
            fp = os.path.join(SU_OUT, "fracSanity", "genes", sub, safe(p) + ".pkl")
            if os.path.exists(fp):
                comp.loc[p, f"fracSanity on the F^K-weighted pool: {k}"] = load_pickle(fp)["F"]
    for k, fs in fits.items():
        if len(fs) == 8:
            comp[f"fracSanity per fraction, summed: {k}"] = pd.Series({p: sum(fs[c]["F"] for c in m) for p, m in POOLS.items()})
    pool_rows, pool_alpha = [], {}
    for k, sub in POOL_FITS.items():
        for p in POOLS:
            fp = os.path.join(SU_OUT, "fracSanity", "genes", sub, safe(p) + ".pkl")
            if not os.path.exists(fp):
                continue
            f = load_pickle(fp)
            comp.loc[p, f"fracSanity on the pool: {k}"] = f["F"]
            a = pd.DataFrame({"alpha": f["alpha_g"], "lo": f["alpha_g_CI_lower"], "hi": f["alpha_g_CI_upper"], "lm": f["log2_alpha_mean"],
                              "outlier_prob": f.get("outlier_prob", np.full(len(f["gene_ids"]), np.nan))}, index=f["gene_ids"])
            pool_alpha[(k, p)] = a
            g = Gp.intersection(a.index)
            t = alpha_mb.loc[g, p]
            r_fit, r_true = a.loc[g, "lm"] - a.loc[g, "lm"].median(), np.log2(t) - np.log2(t).median()
            pool_rows.append({"fit": k, "pool": p, "F_spike": F_true[p], "F_mb": F_mbp[p], "F": f["F"], "F_CI_lower": f["F_CI_lower"], "F_CI_upper": f["F_CI_upper"],
                              "F_identifiable": f["F_identifiable"], "pi_outlier": f["pi_outlier"], "a": f["a"], "b": f["b"],
                              "log2_F_over_F_spike": np.log2(f["F"] / F_true[p]), "log2_F_over_F_mb": np.log2(f["F"] / F_mbp[p]),
                              "bias_log2_vs_alpha_mb": float(np.median(np.log2(a.loc[g, "alpha"]) - np.log2(t))),
                              "coverage_95_alpha_mb": float(((t >= a.loc[g, "lo"]) & (t <= a.loc[g, "hi"])).mean()),
                              "slope_r_on_r_mb": float(np.polyfit(r_true, r_fit, 1)[0]),
                              "share_alpha_mb_above_1": float((t > 1).mean()),
                              "share_outlier_prob_above_0.5": float((a.loc[g, "outlier_prob"] > 0.5).mean())})
    save_tsv(comp.rename_axis("pool").reset_index(), "model", "F_per_compartment.tsv")
    print("F per compartment"); display(comp.round(3))
    pool_res = pd.DataFrame(pool_rows)
    if len(pool_res):
        save_tsv(pool_res, "model", "pooled_compartments.fit_vs_mass_balance.tsv")
        display(pool_res.round(3))
        sns.set(font_scale=0.85, style="ticks")
        fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
        xs = np.linspace(-14, 1, 400)
        for j, p in enumerate(POOLS):
            ax = axes[0, j]
            t = np.log2(alpha_mb[p])
            ax.hist(t.clip(-14, 2), bins=np.linspace(-14, 2, 81), density=True, color="lightgrey", label=r"$\alpha^{mb}$ (pooled mass balance)")
            for k in POOL_FITS:
                if (k, p) in pool_alpha:
                    r = pool_res.set_index(["fit", "pool"]).loc[(k, p)]
                    ax.plot(xs[xs < 0], beta_dist.pdf(2.0 ** xs[xs < 0], r["a"], r["b"]) * 2.0 ** xs[xs < 0] * np.log(2), color=FIT_COLORS[k],
                            lw=1.2, label=f"Beta, {k}")
            ax.axvline(np.log2(F_mbp[p]), color="black", lw=0.8, ls="--")
            ax.set(title=f"{p} ({'+'.join(POOLS[p])})", xlabel=r"log2 $\alpha$", xlim=(-10, 2))
            ax = axes[1, j]
            if ("uniform prior", p) in pool_alpha:
                a = pool_alpha[("uniform prior", p)]
                g = Gp.intersection(a.index)
                ax.hexbin(np.log2(alpha_mb.loc[g, p]), np.log2(a.loc[g, "alpha"]), gridsize=40, mincnt=1, bins="log", cmap="viridis",
                          extent=(-10, 2, -10, 2))
                ax.plot([-10, 2], [-10, 2], color="red", lw=0.6)
                r = pool_res.set_index(["fit", "pool"]).loc[("uniform prior", p)]
                ax.set(title=f"F {r['F']:.3f} (spike-in {r['F_spike']:.3f}, mass balance {r['F_mb']:.3f})\ncov {r['coverage_95_alpha_mb']:.2f}, slope {r['slope_r_on_r_mb']:.2f}",
                       xlabel=r"log2 $\alpha^{mb}$", ylabel=r"log2 $\alpha_g$, uniform prior")
        axes[0, 0].legend(fontsize=6, frameon=False)
        sns.despine(); fig.tight_layout()
        savefig(fig, "model", "pooled_compartments")
        plt.show()
else:
    print("pooled libraries not processed yet (pipeline steps pooled_auth, mass_balance_pooled_auth)")

### Interpretation of the diagnostics

**With the authors' $F$ as the reference, $\alpha^T$ behaves as a share of molecules should.** $\alpha^T_g = F^{\text{spike}}_k 2^{D_{g,k}}$ exceeds 1 for at most 0.1% of the genes in single fractions. In the pooled cytosol, its median is 0.63 and its 99th percentile 1.00: most mRNAs are mainly cytosolic, and the most cytosolic ones are entirely so. Nothing in the construction forces this, so it is an independent check of $F^{\text{spike}}$.

**The shape of $\log_2\alpha$ is the cause of the failure.** A Beta always gives a left-skewed $\log\alpha$, and for a given spread its skewness is bounded (left panel, coloured region). The gradient fractions fall in two groups, and neither is the shape fracSanity assumes:

- **Fractions 1–5, Membranes, Nuclear: right-skewed $\log_2\alpha^T$** (skewness +0.57 to +1.56; fitted Betas −0.66 to −1.38). These fractions hold a minority of strongly enriched RNAs (ER-translated mRNAs and mito RNAs in 1–3; snoRNAs, MALAT1, NEAT1, 7SK in 4–5) on top of a background of all other mRNAs. No Beta has a right tail in $\log\alpha$. The closest is the Gamma limit $b \to \infty$, where only $bF$ is identified. The likelihood then runs to $F \to 0$, and the enriched minority becomes "outliers" (π = 0.5, 31–39% of genes with outlier probability > 0.5). Under this default error model the likelihood is shallow: $F^{\text{spike}}$ costs only 8–15 nats against the maximum. Bounding the outliers (low π) makes it steep (27–310 nats), because the enriched genes now have to be explained by the Beta.
- **Fractions 6–8, Cytosol: strongly left-skewed but narrow** (skewness −1.5 to −2.1 at sd 0.6–1.0 log2). A Beta reaches this corner only with mean $\alpha$ close to 1 (yellow in the left panel). The sharp upper edge of $D_g$ is therefore read as $\alpha = 1$, although $\alpha^T$ of single fractions stays below 0.61 for 99% of the genes. $F$ is set so that the edge sits at 1: 0.64 / 0.55 / 0.69 against 0.12 / 0.31 / 0.26 (authors). The three sum to 1.88, which is impossible (authors: 0.69). Here the data reject $F^{\text{spike}}$ strongly (25–437 nats), and even a tight prior around it does not hold.

**The simulation shows that the shape alone explains the failure.** The Total library and its errors were kept, and a fraction was simulated with known $F$ and two kinds of $\alpha$. The known $F$ values are $F^K$ from the first version of this analysis; any known value serves the purpose.

- **Beta-shaped $\alpha$ (Beta fitted to $\alpha^T$):** $F$ is recovered in all 11 cases. The true $F$ lies in the 95% CI in 10 cases; the exception is Membranes, 0.392 against 0.406, which is just outside a very narrow CI. The $\alpha_g$ bias is ≤ 0.37 log2, coverage 0.91–1.00, slope of $r_g$ on the true $r_g$ 0.77–0.99, and the outlier share ≈ 0.
- **The real $\alpha^T$ shape, same $F$:** the real fits are reproduced. $F \to$ 0.006–0.015 for 1–5, Membranes and Nuclear; 0.55–0.79 for 6–8; Cytosol 0.64 instead of 0.38. The $\alpha_g$ coverage is 0–0.19, and the slopes are 0.16–0.47 for the membrane and nuclear fractions.

So neither the optimiser, the measurement noise, nor the error model breaks the fit. It is the Beta prior on $\alpha_g$: with the real distribution of $\alpha$ the likelihood is maximised at the wrong $F$.

**Pooled compartments: the complete cytosol gives the right $F$; membranes and nuclear do not.** Pools weighted by the authors' $F$ (pipeline step `pooled_auth`):

| compartment | authors' $F$ (sum of the fractions) | mass balance (pooled) | fit, uniform prior | fit, low π |
|:---|---:|---:|---:|---:|
| Membranes (1–3) | 0.184 | 0.206 | 0.005, not identified | 0.0002 |
| Nuclear (4–5) | 0.124 | 0.147 | 0.002, not identified | 0.0009 |
| Cytosol (6–8) | 0.692 | 0.729 | 0.684 [0.661, 0.711] | 0.637 [0.617, 0.659] |

- **Cytosol:** the fit is within −0.02 log2 of the authors' $F$ (uniform prior) and −0.12 log2 (low π). The $\alpha_g$ coverage against $\alpha^{mb}$ is 0.89 / 0.87, and the shrinkage slope 0.77 / 0.80. This is the one case where $\alpha$ reaches 1 for many genes, which is what the Beta needs. With the pool weighted by the read-only $F^K$ (first version), the same fit gave 0.82: the pool must be composed in the right proportions.
- **Membranes and Nuclear:** even when complete, these compartments hold the bulk of genes at a low background level (median $\alpha^T$ 0.11 / 0.06) plus an enriched minority. That is again a right-skewed $\log\alpha$, and $F$ collapses.

**Three independent estimates of the compartments agree:** the authors' $F$ (0.18 / 0.12 / 0.69), the pooled mass balance normalised to sum 1 (0.19 / 0.14 / 0.67), and the sum of the 8-fraction NNLS (0.20 / 0.13 / 0.73). They differ by at most 0.13 log2. The unnormalised mass balance sums to 1.08 (pools) or 1.05 (8 fractions). With the authors' $F$, the predicted Total falls short of the measured Total by a median of 0.23 log2: the gradient recovers about 85% of a typical gene's RNA, and the NNLS absorbs the loss by inflating $F$.

**What this means for fracSanity.**
1. $F$ (and the absolute $\alpha_g$) can be trusted only when the compartment holds most molecules of a large share of genes, so that the upper edge of $D_g$ really is $\alpha = 1$. This holds for the complete cytosol, but not for a minor organelle, one fraction of several, or a FACS subset. For those, $F$ must come from outside: spike-ins with RNA yields, a mass balance against the Total, or spike-ins added per cell or per fraction volume.
2. Bounding the outlier share (low π) removes the outlier discounting: the outlier share falls from 0.32 to 0.03, and the slope of $r_g$ in fractions 1–5 rises from 0.20–0.50 to 0.61–0.67. But it does not fix $F$; in the membrane fractions it makes $F$ even smaller.
3. Possible changes in Sanity_utils:
   - a prior for $\alpha$ that can be right-skewed in $\log\alpha$: a two-component Beta mixture (background + enriched), or a scaled Beta with a free upper bound $\alpha_{\max} < 1$;
   - an option to fix $F$ (from spike-ins or a mass balance) and fit only the gene-level part;
   - a per-gene posterior that ignores the outlier component.

## Per-library fits: each fraction library against the Total library of the same replicate

In [ ]:
rep_dir = os.path.join(SU_OUT, "fracSanity", "genes", "libraries")
rep_rows = []
for _, r in meta.loc[meta["cohort"] != "Total"].iterrows():
    p = os.path.join(rep_dir, f"{safe(r['sample'])}.pkl")
    if not os.path.exists(p):
        continue
    f = load_pickle(p)
    rep_rows.append({"sample": r["sample"], "cohort": r["cohort"], "replicate": r["replicate"], "F": f["F"], "F_CI_lower": f["F_CI_lower"],
                     "F_CI_upper": f["F_CI_upper"], "F_identifiable": f["F_identifiable"], "pi_outlier": f["pi_outlier"],
                     "F_spike": F_sp.loc[r["cohort"], r["replicate"]], "F_mb": mb_rep.loc[r["cohort"], r["replicate"]]})
rep_res = pd.DataFrame(rep_rows)
save_tsv(rep_res, "F", "F_per_library_vs_spike_in.tsv")
display(rep_res.round(4))
if len(rep_res):
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    for _, r in rep_res.iterrows():
        ax.errorbar(r["F_spike"], r["F"], yerr=[[r["F"] - r["F_CI_lower"]], [r["F_CI_upper"] - r["F"]]], fmt=REP_MARKERS[r["replicate"]],
                    color=FRAC_COLORS[r["cohort"]], ms=5, mec="black", mew=0.4, capsize=2, elinewidth=0.6)
    lim = [1e-5, 1]
    ax.plot(lim, lim, color="grey", ls="--", lw=0.8)
    ax.set(xscale="log", yscale="log", xlim=(min(rep_res["F_spike"].min() / 2, 1e-2), 1), ylim=lim,
           xlabel="$F^{spike}$ (library)", ylabel="fracSanity $F$ (uniform prior), 95% CI")
    ax.legend(handles=[plt.Line2D([], [], color=FRAC_COLORS[c], marker="o", ls="", label=c) for c in COHORTS], fontsize=6, frameon=False,
              loc="upper left", bbox_to_anchor=(1, 1))
    sns.despine(); fig.tight_layout()
    savefig(fig, "F", "F_per_library_vs_spike_in")
    plt.show()

### Interpretation of the test

- **Gene ranking: correct in every fraction and every fit.** Spearman of $\alpha_g$ with $\alpha^T$ is 0.94–0.997, and that of $r_g$ with $r^T_g$ 0.97–0.996. This is expected: within a fraction, the ranking of $\alpha_g$ is that of the centred $D_g$. Spearman with $\alpha^{\Sigma}$ (normalised to the sum over the 8 fractions) is lower, 0.74–0.90: the sum over fractions is not the same for every gene (recovery).
- **$F$: none of the fits without the spike-ins recovers it.**
  - **Uniform prior:** $F \to$ 0.002–0.005 in fractions 1–5 (not identifiable; authors 0.04–0.08). In fractions 6–8 it is 0.64 / 0.55 / 0.69 against 0.12 / 0.31 / 0.26, i.e. +2.4 / +0.8 / +1.4 log2.
  - **Mass-balance prior:** moves $F$ only in F8 (0.18), where the likelihood is flat and the 8-fraction NNLS is uninformative (it is 0 for F1, F2, F8, because neighbouring fractions are collinear).
  - **Spike-in prior** (Beta around the authors' $F$, ±25%): in fractions 1–5 the likelihood is shallow (8–15 nats), so the prior holds $F$ (−0.18 to −0.31 log2). In 6–8 the data override it (0.38 / 0.55 / 0.34).
- **$\alpha_g$ when $F$ is right:** with the spike-in prior in fractions 1–5, the absolute $\alpha_g$ agrees with $\alpha^T$: bias −0.07 to −0.25 log2, coverage 0.79–0.95. This tests the gene-level part of the model given $F$. $\alpha^T$ uses the same $D_g$, so it measures shrinkage, not noise. The ranking is still compressed: the slope of $r_g$ on $r^T_g$ is 0.20–0.50, because 31–39% of genes, the enriched ones, are treated as outliers.
- **With the outliers bounded (low π):** the slope rises to 0.61–0.67 (fractions 1–5) and the outlier share falls to 1–3%. But the likelihood then rejects $F^{\text{spike}}$ (27–310 nats), and even the spike-in prior is pulled down to 0.014–0.036 (bias −0.3 to −2.3 log2). In fractions 1–5, no combination of prior and error model gives both the right $F$ and an unshrunk ranking.
- **$F^K$ prior** (first version, equal SIRV amounts assumed): bias +0.2 to +1.2 log2 in fractions 1–5, because $F^K$ is not the RNA content.
- **Per-library fits** (each fraction library against the Total of its replicate) show the same picture per replicate.
  - $F$ is identifiable only for the cytosolic fractions: 8 of 24 libraries, all of 6–8 except F6 R2.
  - The median $F$ is 0.34 / 0.64 / 0.79 for 6 / 7 / 8, against 0.09 / 0.31 / 0.26 (authors).
  - For 1–5, $F \to$ 0.005–0.020, against 0.03–0.11.

  The failure is therefore not caused by averaging replicates.

## The gradient fractions compared with the FACS-sorted ONT fractions and APEX-seq

The F-free ranking $r_g = \log_2\alpha_g - \text{median}$ (uniform-prior fits) of the 8 gradient fractions (U2OS) against the same quantity of the FACS-sorted ONT fractions (HCT116; `ONT_organelles.fracSanity.ipynb`) and of the APEX-seq constructs (HEK293T; `APEX_seq.fracSanity.ipynb`, if already run). Protein-coding genes with ≥ 10 TPM (gradient Total), ≥ 10 CPM (ONT Total) and ≥ 10 TPM (APEX-seq controls). Expected pairs: ER / TIS ↔ fractions 1–3 and APEX ERM / KDEL; Free ↔ cytosolic fractions 6–8 and APEX NES; the nuclear contamination of Golgi, PEX, Endo and TJ ↔ fraction 4 (nucleolus) and APEX NLS / NIK.

In [ ]:
def load_r(folder, min_expr, expr_col):
    f = os.path.join(folder, "alpha_genes", "fracSanity_alpha.genes.per_cohort.tsv.gz")
    if not os.path.exists(f):
        return None
    a = pd.read_csv(f, sep="\t")
    a = a.loc[~a["gene_id"].str.endswith("_PAR_Y") & (a["gene_type"] == "protein_coding") & (a[expr_col] >= min_expr)]
    a["gene"] = a["gene_id"].str.replace(r"\.\d+$", "", regex=True)
    return a.pivot_table(index="gene", columns="cohort", values="r_rel", aggfunc="first")

ok = "uniform prior" in fits and len(fits["uniform prior"]) == 8
if ok:
    r_lo = pd.DataFrame({c: alpha_fit[("uniform prior", c)]["log2_alpha_mean"] for c in COHORTS})
    r_lo = r_lo - r_lo.median()
    keep = r_lo.index[(total_tpm.reindex(r_lo.index) >= MIN_TPM).to_numpy() & (biotype.reindex(r_lo.index) == "protein_coding").to_numpy()]
    r_lo = r_lo.loc[keep]
    r_lo.index = r_lo.index.str.replace(r"\.\d+$", "", regex=True)
    r_lo.columns = [f"{c} {LABELS[c]}" for c in COHORTS]
    others = {"ONT (HCT116, FACS)": load_r(T_ONT, 10, "Total_CPM"), "APEX-seq (HEK293T)": load_r(T_APEX, 10, "Total_TPM")}
    others = {k: v for k, v in others.items() if v is not None}
    for k, o in others.items():
        common = r_lo.index.intersection(o.index)
        m = pd.DataFrame({b: [spearmanr(r_lo.loc[common, a], o.loc[common, b], nan_policy="omit")[0] for a in r_lo.columns] for b in o.columns},
                         index=r_lo.columns)
        save_tsv(m.rename_axis("gradient fraction").reset_index(), "comparison", f"spearman_r_rel.gradient_vs_{safe(k.split(' ')[0])}.tsv")
        sns.set(font_scale=0.8, style="white")
        fig, ax = plt.subplots(figsize=(0.55 * m.shape[1] + 3, 4.2))
        lim = np.nanmax(np.abs(m.values))
        sns.heatmap(m, cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, annot=True, fmt=".2f", annot_kws={"fontsize": 6}, ax=ax,
                    cbar_kws={"label": "Spearman of $r_g$"})
        ax.set(title=f"gradient (U2OS) vs {k}: {len(common):,} protein-coding genes", xlabel=k, ylabel="gradient fraction")
        fig.tight_layout()
        savefig(fig, "comparison", f"spearman_r_rel.gradient_vs_{safe(k.split(' ')[0])}")
        plt.show()
    # the gradient fractions among themselves
    m = r_lo.corr(method="spearman")
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(m, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"fontsize": 6}, ax=ax)
    ax.set(title="gradient fractions: Spearman of $r_g$")
    fig.tight_layout()
    savefig(fig, "comparison", "spearman_r_rel.gradient_fractions")
    plt.show()

**Membrane-bound mRNAs of the gradient in the other data sets.** Genome-wide correlations are dominated by the bulk of the mRNAs, which differ little between fractions, and by mRNA length (fractions 7–8 are enriched for short mRNAs, the APEX-seq enrichments of ERM, OMM, LMA, NES rise with length). A sharper test: the 5% of protein-coding genes (≥ 10 TPM in Total) with the highest membrane-to-cytosol ratio of the gradient, $\log_2\left(\sum_{k=1}^{3}\alpha^{\Sigma}_{g,k} / \sum_{k=6}^{8}\alpha^{\Sigma}_{g,k}\right)$ (spike-in $F$; independent of fracSanity), and their $r_g$ in each ONT and APEX-seq fraction: AUC (probability that a gene of the set has a higher $r_g$ than a gene outside it) and the difference of the medians. Also: Spearman of the gradient $r^T_g$ (spike-in) with mRNA length per fraction.

In [ ]:
from scipy.stats import mannwhitneyu
pc_eval = EVAL[(biotype.reindex(EVAL) == "protein_coding").to_numpy()]
mem_cyto = np.log2(alpha_S.loc[pc_eval, ["F1", "F2", "F3"]].sum(axis=1) / alpha_S.loc[pc_eval, ["F6", "F7", "F8"]].sum(axis=1))
mem_cyto.index = mem_cyto.index.str.replace(r"\.\d+$", "", regex=True)
thr = mem_cyto.quantile(0.95)
mem_set = set(mem_cyto.index[mem_cyto >= thr])
names_nov = pd.Series(gene_names.reindex(pc_eval).to_numpy(), index=mem_cyto.index)
print(f"gradient membrane set: {len(mem_set)} genes with log2 membrane / cytosol >= {thr:.2f}; top 30: "
      + ", ".join(names_nov[mem_cyto.sort_values(ascending=False).index[:30]]))
others = {"ONT (HCT116, FACS)": load_r(T_ONT, 10, "Total_CPM"), "APEX-seq (HEK293T)": load_r(T_APEX, 10, "Total_TPM")}
auc_rows = []
for k, o in others.items():
    if o is None:
        continue
    g = mem_cyto.index.intersection(o.index)
    ins = g.isin(list(mem_set))
    for c in o.columns:
        x = o.loc[g, c]
        a, b = x[ins].dropna(), x[~ins].dropna()
        auc_rows.append({"data set": k, "fraction": c, "n_in_set": len(a), "n_other": len(b),
                         "AUC": mannwhitneyu(a, b).statistic / len(a) / len(b), "delta_median_r": a.median() - b.median(),
                         "p_value": mannwhitneyu(a, b).pvalue})
auc = pd.DataFrame(auc_rows)
save_tsv(auc, "comparison", "gradient_membrane_set_in_ONT_and_APEX.tsv")
display(auc.round(3))
if len(auc):
    fig, ax = plt.subplots(figsize=(10, 3.4))
    cols = [FRAC_COLORS["F2"] if v > 0.5 else "tab:grey" for v in auc["AUC"]]
    ax.bar(range(len(auc)), auc["AUC"] - 0.5, bottom=0.5, color=cols)
    ax.axhline(0.5, color="black", lw=0.6)
    ax.set_xticks(range(len(auc)), [f"{d.split(' ')[0]}: {c}" for d, c in zip(auc["data set"], auc["fraction"])], rotation=90, fontsize=7)
    ax.set(ylabel="AUC of the membrane set", ylim=(0, 1), title=f"gradient membrane mRNAs ({len(mem_set)}) in the ONT and APEX-seq fractions")
    sns.despine(); fig.tight_layout()
    savefig(fig, "comparison", "gradient_membrane_set_in_ONT_and_APEX")
    plt.show()

lengths = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_lengths.tsv"), sep="\t", index_col=0)["length"]
len_cor = pd.Series({c: spearmanr(np.log2(alpha_T.loc[pc_eval, c]), lengths.reindex(pc_eval))[0] for c in COHORTS},
                    name="Spearman of log2 alpha^T with gene length (protein-coding)")
display(len_cor.round(2).to_frame().T)

# F7 / F8 against APEX ERM, with and without gene length (partial Spearman: Pearson of rank residuals)
if ok and "APEX-seq (HEK293T)" in others:
    o = others["APEX-seq (HEK293T)"]
    ln = lengths.copy()
    ln.index = ln.index.str.replace(r"\.\d+$", "", regex=True)
    ln = ln.loc[~ln.index.duplicated()]
    common = r_lo.index.intersection(o.index).intersection(ln.index)
    part = []
    for c in [col for col in r_lo.columns if col.startswith(("F7", "F8"))]:
        d = pd.DataFrame({"x": r_lo.loc[common, c], "y": o.loc[common, "APEX: ERM"], "z": np.log2(ln.loc[common])}).dropna().rank()
        res = {k: d[k] - np.polyval(np.polyfit(d["z"], d[k], 1), d["z"]) for k in ("x", "y")}
        part.append({"gradient fraction": c, "n_genes": len(d), "spearman_vs_APEX_ERM": spearmanr(d["x"], d["y"])[0],
                     "partial_spearman_given_length": np.corrcoef(res["x"], res["y"])[0, 1]})
    part = pd.DataFrame(part)
    save_tsv(part, "comparison", "F7_F8_vs_APEX_ERM.partial_given_length.tsv")
    display(part.round(2))

### Interpretation of the comparison

**The membrane / cytosol contrast of the gradient agrees with both other data sets.** The gradient membrane set (343 genes, top 5% of $\log_2\alpha^{\Sigma}$(1–3) / $\alpha^{\Sigma}$(6–8) with the authors' $F$; top genes COL4A1, FN1, TNC, PDIA4, HSPA5, SEL1L, HSP90B1, …: secreted and ER-resident proteins) is enriched in:

| data set | where the set is enriched (AUC) | where it is depleted (AUC) |
|:---|:---|:---|
| APEX-seq (HEK293T) | OMM 0.91, ERM 0.83, Nuc fractionation 0.75 | NLS 0.30, NES 0.31, Cyto fractionation 0.35 |
| ONT, FACS (HCT116) | Golgi 0.72, Endo 0.71, TJ 0.68, PEX 0.67, ER 0.65 | Free 0.14–0.23, TIS 0.22 |

- Three methods in three cell lines (density, proximity labelling, organelle sorting) find the same ER-translated mRNAs at membranes.
- OMM ranking above ERM is consistent with ER–mitochondria contacts and with the mito RNAs in fractions 1–3.
- The enrichment in the APEX nuclear fractionation fits the ER being continuous with the nuclear envelope.
- In ONT, the effect is stronger in Golgi / Endo / TJ / PEX than in the ER fraction itself, as already seen with APEX ERM in `APEX_vs_ONT_organelles.fracSanity.ipynb`: ER-translated mRNAs co-sort with these membranes.

**Genome-wide, the single fractions match less well.** These correlations use the F-free $r_g$, so they do not depend on the choice of $F$.
- **Fractions 7–8 are the clearest anti-membrane signal:**
  - against APEX ERM, Spearman −0.57 / −0.67, and −0.60 against OMM;
  - +0.54 / +0.47 against the APEX cytosolic fractionation;
  - against ONT Golgi / PEX / TJ, −0.47 to −0.53, and +0.11 to +0.34 against Free.
- **Fractions 1–3 alone correlate weakly.**
  - ERM: +0.06 to +0.14.
  - ONT ER: −0.17 / −0.18.
  - In addition, NES / NLS / NucPore: −0.23 to −0.34.

  On their own, fractions 1–3 are dominated by a broad background of all mRNAs. The membrane signal is the minority that the contrast with 6–8 isolates.
- **Fractions 4–5 (nuclear RNPs)** match the APEX nuclear-side constructs:
  - F5 with NIK 0.48, LMA 0.48 and NLS 0.35;
  - F4 with the Nuc fractionation 0.50.
- **Length:** much of the cytosolic ranking follows transcript length. Spearman of $\log_2\alpha^T$ with gene length for protein-coding genes is −0.43 / −0.55 in F7 / F8 and +0.40 / +0.30 in F5 / F6: short mRNAs sediment in 7–8, long ones in 5–6. Given length, the F7 / F8 ↔ ERM correlation weakens to −0.42 / −0.50 (partial Spearman) but remains.

## Summary

1. **The spike-ins do give the RNA content of the fractions, with the authors' measurements.**
   - The SIRVs were added in proportion to the rRNA-depleted RNA of each library, not in the same amount, and every library started from 1 µg. So the SIRV share of the reads alone ($F^K$) is not the RNA content.
   - The authors combine it with the ng SIRVs added and the RNA yield of each fraction (`sirv_correction.rds` of their repository; method and comparison with equal spike-in amounts added before RNA isolation in the section above).
   - Resulting $F^{\text{spike}}$:
     - fractions 1–5: 0.04–0.08;
     - fractions 6 / 7 / 8: 0.12 / 0.31 / 0.26;
     - compartments: membranes 0.18, nuclear 0.12, cytosol 0.69.
   - Validation:
     - Pooled mass balance, normalised: 0.19 / 0.14 / 0.67.
     - Gene-wise mass balance: 89% of genes are reconstructed within 1.5×, against 78–83% for a permuted $F^{\text{spike}}$, $F = 1/8$ or $F^K$.
     - $\alpha^T \le 1$ for ≥ 99.9% of genes in every fraction, and the cytosol's 99th percentile is 1.00.
   - $F^{\text{spike}}$ is a share of non-rRNA mass. Converted to molecules of the analysed genes, single fractions change by up to 25%, and the fit to the mass balance does not improve (87% within 1.5×).
2. **The gene ranking is recovered; the absolute $F$ and $\alpha_g$ are not.**
   - Spearman of $\alpha_g$ with $\alpha^T$ is 0.94–0.997 in every fraction.
   - $F \to 0$ for the membrane and nuclear fractions, 0.002–0.005 against 0.04–0.08.
   - For the cytosolic fractions $F$ is 1.8–5× too high: the three sum to 1.88 against 0.69.
   - Only the complete cytosol, pooled in the right proportions, gives the right $F$: 0.68 against 0.69 (authors) and 0.73 (mass balance).
3. **The cause is the Beta shape of the $\alpha$ prior.**
   - Fractions with an enriched minority over a broad background have a right-skewed $\log\alpha$, which no Beta can produce, so $F \to 0$.
   - Narrow cytosolic fractions put the edge of $D_g$ at $\alpha = 1$, so $F$ comes out too high. There the data reject the true $F$ by 25–437 nats, and a tight prior around it does not hold.
   - A simulation with the same data and known $F$ separates the two: $F$ is recovered when $\alpha$ is Beta-shaped, and the real failure is reproduced when $\alpha$ has the real shape.
4. **Given the right $F$, the gene-level part is nearly right on the absolute scale, but the ranking is shrunk.**
   - In fractions 1–5 with the spike-in prior, the $\alpha_g$ bias is −0.07 to −0.25 log2 and the coverage 0.79–0.95.
   - But the slope of $r_g$ is only 0.20–0.50, because the enriched genes are discounted as outliers.
   - Bounding the outlier share restores the slope (0.61–0.67) but pulls $F$ away from the truth again.
5. **Consequence for the ONT and APEX-seq fractions:** their $F$ and absolute $\alpha_g$ are not reliable, for the same reason: a minor organelle or construct, measured against Total, has an enriched minority over a background. Their $r_g$ rankings are reliable, apart from the shrinkage from outlier discounting, which the low-π error model removes.
6. **Cross-data agreement:**
   - The gradient membrane / cytosol contrast picks the ER-translated mRNAs. These are enriched in APEX OMM / ERM (AUC 0.91 / 0.83) and in the ONT Golgi, Endo, TJ, PEX and ER fractions (0.65–0.72), and depleted in ONT Free (0.14–0.23).
   - The cytosolic fractions 7–8 anticorrelate with APEX ERM genome-wide (−0.57 / −0.67), partly through transcript length.
7. **Suggested changes to fracSanity (Sanity_utils):**
   - an $\alpha$ prior that allows a right-skewed $\log\alpha$: a Beta mixture (background + enriched component), or a scaled Beta with a free $\alpha_{\max}$;
   - an option to fix $F$ from outside (spike-ins with RNA yields, or a mass balance when the complementary fractions are measured);
   - with $F$ fixed, a per-gene posterior that does not discount localised genes as outliers.